# CUDA replication of the PhysicsNeMo physics-informed Darcy FNO study (Kaggle GPU)

Launcher for [physicsnemo-pino-darcy](https://github.com/AdebanjiAdelowo/physicsnemo-pino-darcy). It checks out one commit of the repository and calls its scripts; it contains no model or training code. The analysis plan, including the replication criterion, is [`docs/CUDA_REPLICATION_PLAN.md`](https://github.com/AdebanjiAdelowo/physicsnemo-pino-darcy/blob/main/docs/CUDA_REPLICATION_PLAN.md).

Session settings: **Accelerator** an NVIDIA GPU, **Internet** on (the dataset is downloaded from Google Drive, about 0.8 GB).

| Step | What runs | When |
|---|---|---|
| A | checkout and its verification, GPU and dependency check, dataset download and data check, tests, CUDA smoke study, one-epoch timing probe | always |
| B | training of the studies selected by `PLAN`, each with `--resume` | only if `PLAN` is set and the projection fits the session budget |
| C | audit evaluations, archives, and the replication analysis | after step B |

| `PLAN` | Studies (configs) | Runs |
|---|---|---|
| `None` (default) | none: step A only | 0 |
| `"minimal"` | `full_cuda` (seeds 0 to 2) + `full_cuda_more` (seeds 3, 4): data only, `shipped` 0.1, `consistent` 0.1 | 15 |
| `"g2"` | `full_cuda` + `full_cuda_sweep` (weights 0.001, 0.01, 1): the 27 runs of `configs/full.yaml` | 27 |
| `"all"` | all three | 33 |

The CUDA studies have their own names and directories (`runs/full_cuda*`, `results/full_cuda*`); nothing here reads or writes an MPS study. If the projection does not fit, step B is **not started and nothing is reduced**. To continue in a new session, attach the archives of the earlier session as an input and set `RESTORE_DIR`: finished runs are restored and skipped, and nothing that exists is overwritten.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-pino-darcy.git"
REF = "SET_TO_FULL_40_CHARACTER_COMMIT_SHA"   # output of `git rev-parse HEAD` for the commit to run
PLAN = None                                   # None (step A only), "minimal" (15 runs), "g2" (27 runs) or "all" (33 runs)
SESSION_BUDGET_HOURS = 9.0                    # GPU time you are willing to spend in this session; check Kaggle's current limits
RESTORE_DIR = None                            # input directory holding the archives of an earlier session, to resume
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

PLANS = {
    "minimal": ["full_cuda", "full_cuda_more"],
    "g2": ["full_cuda", "full_cuda_sweep"],
    "all": ["full_cuda", "full_cuda_more", "full_cuda_sweep"],
}
import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
assert PLAN is None or PLAN in PLANS, f"PLAN must be None or one of {sorted(PLANS)}"
STUDIES = PLANS[PLAN] if PLAN else []
assert all(s.startswith("full_cuda") for s in STUDIES), "only the CUDA studies may be trained here"
WORK = "/kaggle/working"
REPO = f"{WORK}/physicsnemo-pino-darcy"

def hours_left():
    return SESSION_BUDGET_HOURS - (time.time() - T0) / 3600

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "CUDA is not available. Select a GPU accelerator; this notebook does not fall back to CPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import os, subprocess
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def tree_is_clean():
    return not subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert tree_is_clean(), "the working tree is not clean"
for name in STUDIES:
    assert os.path.exists(f"{REPO}/configs/{name}.yaml"), f"configs/{name}.yaml is not in commit {REF}"
print("commit", head)

The requirements pin PhysicsNeMo to a commit of its main branch. `pip` may replace the PyTorch build of the image if it is older than PhysicsNeMo requires; the next cell prints the versions afterwards, in a new process, checks CUDA again and checks that the pinned PhysicsNeMo commit is the one installed.

In [ ]:
sh("pip install -q -r requirements.txt", cwd=REPO)
sh('python -c "import torch, physicsnemo, warp, sympy; assert torch.cuda.is_available(); '
   "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, '| warp', warp.__version__, '| sympy', sympy.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)
sh('python -c "import sys; sys.path.insert(0, \'src\'); from pino_darcy.provenance import physicsnemo_source, UPSTREAM_COMMIT; '
   "s = physicsnemo_source(); print('physicsnemo source', s); assert s['commit'] == UPSTREAM_COMMIT, 'installed PhysicsNeMo is not the pinned commit'\"", cwd=REPO)

## Step A: dataset, data check, tests, smoke study and timing probe

Downloads `Darcy_241.zip`, writes the training, validation and test splits with their checksums, and checks what the data represent. The check runs on CPU, writes outside the clone, and must give the dataset files recorded in the repository.

In [ ]:
import json, zipfile
sh("python scripts/prepare_data.py", cwd=REPO)
CHECK = f"{WORK}/data-check"   # outside the clone, so the tree stays clean for the study
sh(f"python scripts/verify_data.py --out-root {CHECK}", cwd=REPO)
check = json.load(open(f"{CHECK}/results/data_check.json"))
local = json.load(open(f"{REPO}/results/data_check.json"))
print("archive sha256:", check["dataset"]["archive_sha256"])
assert check["dataset"]["files"] == local["dataset"]["files"], "the prepared dataset differs from the tracked data check"
print("DATASET IDENTICAL TO THE TRACKED CHECK: True")
with zipfile.ZipFile(f"{WORK}/physicsnemo-pino-darcy-data-check.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for rel in ("results/data_check.json", "figures/data_check.png"):
        z.write(f"{CHECK}/{rel}", rel)
assert tree_is_clean(), "step A modified the clone"

In [ ]:
sh("python -m pytest -q", cwd=REPO)

In [ ]:
# CUDA smoke study: three tiny models, two epochs on eight samples. The numbers are not results.
sh("python scripts/train.py --config smoke device=cuda name=smoke_cuda", cwd=REPO)

### Timing probe

Trains the model of the full study for one epoch, once without and once with the PDE loss. The projection below extrapolates these two epochs to the unfinished runs of every plan (also with `PLAN = None`, so that the first session shows what each plan costs) and adds nothing for evaluation or setup.

In [ ]:
sh('python scripts/train.py --config full --no-evaluate name=full_probe max_epochs=1 "study.seeds=[0]" '
   '"study.residuals=[consistent]" "study.physics_weights=[0,0.1]"', cwd=REPO)
t_data = json.load(open(f"{REPO}/runs/full_probe/dataonly_seed0/train_metrics.json"))["train_seconds"]
t_pde = json.load(open(f"{REPO}/runs/full_probe/consistent_w0.1_seed0/train_metrics.json"))["train_seconds"]
print(f"one epoch: {t_data:.1f} s data-only, {t_pde:.1f} s with the PDE loss, on {torch.cuda.get_device_name(0)}")

## Step B: training

`RESTORE_DIR` (optional) restores the archives of an earlier session: the records into `runs/` and the final checkpoints from the eval bundles. A file that already exists is never replaced. Every restored finished run must have its final checkpoint.

In [ ]:
import glob
if RESTORE_DIR:
    for path in sorted(glob.glob(f"{RESTORE_DIR}/**/physicsnemo-pino-darcy-full_cuda*.zip", recursive=True)):
        name = os.path.basename(path)
        if name.endswith("-audit.zip"):
            continue   # audit files are recomputed from the checkpoints
        target = REPO if name.endswith("-eval-bundle.zip") else f"{REPO}/runs"   # bundles hold runs/<study>/..., records <study>/...
        with zipfile.ZipFile(path) as z:
            new = [m for m in z.namelist() if not os.path.exists(os.path.join(target, m))]
            z.extractall(target, new)
        print(f"restored {len(new)} files from {name}")
    assert tree_is_clean(), "restoring modified tracked files"

# status of every CUDA study, whatever PLAN is, so that a PLAN = None session reports the projection of each plan
ALL_STUDIES = sorted({s for studies in PLANS.values() for s in studies})
out = subprocess.run(["python", "scripts/study_status.py", *ALL_STUDIES], cwd=REPO, capture_output=True, text=True, check=True).stdout
status = json.loads(out)
hours = {}
for study, st in status.items():
    assert st["device"] == "cuda" and st["output_dir"] == f"runs/{study}", st
    assert not st["missing_checkpoint"], f"{study}: finished runs without their final checkpoint: {st['missing_checkpoint']}; restore their eval bundle"
    hours[study] = sum((t_data if r["physics_weight"] == 0 else t_pde) * st["max_epochs"] for r in st["to_train"]) / 3600
    print(f"{study}: {st['runs']} runs, {len(st['finished'])} finished, {len(st['to_train'])} to train, projected {hours[study]:.1f} h")
for plan, studies in PLANS.items():
    n_todo = sum(len(status[s]["to_train"]) for s in studies)
    print(f"  PLAN = {plan!r}: {sum(status[s]['runs'] for s in studies)} runs, {n_todo} to train, projected {sum(hours[s] for s in studies):.1f} h of training")
projection_h = sum(hours[s] for s in STUDIES)
print(f"PROJECTION for step B (PLAN = {PLAN!r}): {projection_h:.1f} h. Left of the session budget: {hours_left():.1f} h.")

In [ ]:
trained = False
if PLAN is None:
    print("Step B skipped: PLAN is None. Set PLAN to 'minimal', 'g2' or 'all' to train.")
elif projection_h > hours_left():
    print(f"STEP B NOT STARTED: projected {projection_h:.1f} h, {hours_left():.1f} h left of SESSION_BUDGET_HOURS = {SESSION_BUDGET_HOURS}.")
    print("No configuration was reduced. Run fewer studies per session (finished runs are kept by RESTORE_DIR) or raise the budget.")
else:
    for study in STUDIES:
        sh(f"python scripts/train.py --config {study} --resume", cwd=REPO)   # skips finished runs; refuses a changed configuration
        # archive after every study, so that a session that ends early keeps what is finished
        sh(f"python scripts/results.py package runs/{study}", cwd=REPO)
        sh(f"python scripts/results.py bundle runs/{study}", cwd=REPO)
        sh(f"cp runs/{study}.zip {WORK}/physicsnemo-pino-darcy-{study}.zip && cp runs/{study}_eval_bundle.zip {WORK}/physicsnemo-pino-darcy-{study}-eval-bundle.zip", cwd=REPO)
    trained = True
    print("STEP B COMPLETE.")

## Step C: audit evaluations, archives and the replication analysis

Run after all training of the session, so that their output files cannot mark a study as trained from a modified tree. The analysis publishes the studies into the clone's `results/full_cuda*` (never an MPS directory) and evaluates the pre-registered criterion; its output is written to `/kaggle/working`.

In [ ]:
if trained:
    for study in STUDIES:
        sh(f"python scripts/evaluate_solver_residual.py --config {study}", cwd=REPO)
        sh(f"python scripts/evaluate_generalisation.py --config {study}", cwd=REPO)
        with zipfile.ZipFile(f"{WORK}/physicsnemo-pino-darcy-{study}-audit.zip", "w", zipfile.ZIP_DEFLATED) as z:
            for path in sorted(glob.glob(f"{REPO}/results/audit/{study}_*.json")):
                z.write(path, os.path.relpath(path, REPO))
        assert not os.path.exists(f"{REPO}/results/{study}"), f"results/{study} exists in the clone"
        sh(f"python scripts/results.py publish runs/{study}", cwd=REPO)
    sh(f"python scripts/cuda_replication.py --out {WORK}/cuda_replication.json --figure {WORK}/cuda_replication.png", cwd=REPO)

In [ ]:
print("Archives in /kaggle/working:")
for path in sorted(glob.glob(f"{WORK}/physicsnemo-pino-darcy-*.zip")) + sorted(glob.glob(f"{WORK}/cuda_replication.*")):
    print(f"  {path}  {os.path.getsize(path) / 2**20:.1f} MB")
print(f"Elapsed: {(time.time() - T0) / 3600:.2f} h. PLAN: {PLAN}. Step B: {'done' if trained else 'not run'}.")
if not trained:
    print("KAGGLE CUDA CHECKS COMPLETE. No study was trained.")